# NB0b — Cleaning & selecting the final 3,500 human articles

NB0 harvested 15,000 raw political / Middle-East articles from CulturaX. This notebook turns that
pool into my final human corpus of 3,500, in three stages:

1. **Light cleaning + reject-on-artifact** — lightly strip obvious wrappers, then DROP any article
   that still shows a scraping artifact (I have abundance, so I keep only the perfectly clean ones).
   I never strip digits/punctuation from a body — that raw punctuation is exactly why I re-harvested.
2. **Topic filtering (medium strictness)** — keep genuine political / public-affairs /
   political-economy hard news; drop the culture / sports / health / opinion pieces that a loose
   keyword match let through.
3. **Selection of 3,500** — natural source distribution (no per-source cap, as decided), a healthy
   length spread, and content-level dedup.

**Input:** `culturax_raw_harvest.parquet` (15,000).
**Output:** `ha_corpus.parquet` (3,500) — the human half of the dataset.

In [1]:
import pandas as pd, numpy as np, re
h = pd.read_parquet('/kaggle/input/notebooks/bahaaqassem/nb0-harvest-culturax/culturax_raw_harvest.parquet')
print('input:', h.shape)
print('sources:\n', h['source_domain'].value_counts())

input: (15000, 6)
sources:
 source_domain
aljazeera.net     7903
alaraby.co.uk     2731
arabi21.com       1259
alarabiya.net     1117
arabic.rt.com      670
almayadeen.net     470
maannews.net       341
alquds.co.uk       217
qudspress.com      182
aa.com.tr           52
wafa.ps             42
aljazeera.com        8
alaraby.tv           8
Name: count, dtype: int64


## Stage 1 — light cleaning + reject-on-artifact

Instead of chasing every scraping artifact with complex substitutions (which risks either eating
real content or leaving residue), I flip the approach: I do a **light** clean of the obvious
wrappers, then I **reject any article that still shows a known artifact** and move on. I harvested
15k and only need 3,500 — abundance lets me keep only the ones that come out perfectly clean.

Two artifact tiers:
- **Line-level wrappers** removed in `light_clean` (hijri/day headers, share buttons, pipe tails,
  photo captions).
- **Reject markers** in `has_artifact` — if any of these survive, the whole article is dropped:
  a stray `|` (not a month separator), `آخر تحديث`, `غرينتش`, `بتوقيت`, `GMT`, a UUID, "منذ N
  ساعات", a long Latin run, a timestamp `الساعة HH:MM (`, or "نشر في:".

I print the source distribution before and after rejection so the effect is transparent — some
outlets (e.g. RT Arabic, whose every article carries a `GMT` timestamp) are largely dropped. That
is an accepted, documented characteristic of the corpus: the thesis separates human from AI, not
one outlet from another, and I keep raw punctuation intact (which is the whole point).

In [2]:
DAYS = r'(?:الأحد|الاثنين|الإثنين|الثلاثاء|الأربعاء|الخميس|الجمعة|السبت)'
MONTHS = (r'(?:يناير|فبراير|مارس|أبريل|إبريل|مايو|يونيو|يوليو|أغسطس|سبتمبر|أكتوبر|اكتوبر|'
          r'نوفمبر|ديسمبر|شباط|آذار|نيسان|أيار|حزيران|تموز|آب|أيلول|تشرين|كانون)')
UI_WORDS = {'شارك','البث الحي','اطبع','أرسل','تكبير الخط','تصغير الخط','follow','تابعنا','انسخ الرابط',''}

def light_clean(text):
    lines = text.split('\n')
    out = []
    for ln in lines:
        s = ln.strip()
        if re.match(rf'^{DAYS}\s+\d{{1,4}}[/\d]+\s*هـ', s):     # hijri header
            continue
        if re.match(rf'^{DAYS}،?\s+\d{{1,2}}[/\s]', s):          # day + date header
            continue
        if s in UI_WORDS:
            continue
        out.append(ln)
    text = '\n'.join(out)
    # pipe tails, protecting month separators (حزيران|يونيو)
    text = re.sub(rf'({MONTHS})\|({MONTHS})', r'\1␟\2', text)
    text = re.sub(r'\s+\|.*$', ' ', text, flags=re.M)
    text = re.sub(r'\|\s+.*$', ' ', text, flags=re.M)
    text = text.replace('␟', '|')
    # photo/agency captions
    text = re.sub(r'\((?:الفرنسية|الأناضول|رويترز[^)]{0,20}|Getty[^)]{0,15}|غيتي[^)]{0,10}|'
                  r'الجزيرة(?:\s*نت)?|أ\s?ف\s?ب|أرشيف)\)', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n', text)
    return text.strip()

def has_artifact(t):
    # if ANY of these survive, I reject the whole article and move on
    if re.search(rf'\|(?!{MONTHS})', t) and not re.search(rf'{MONTHS}\|{MONTHS}', t):
        return True
    for kw in ('آخر تحديث', 'غرينتش', 'بتوقيت', 'GMT'):
        if kw in t:
            return True
    if re.search(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}', t):        # UUID
        return True
    if re.search(r'منذ\s+\d+\s+(?:ساعة|ساعات|دقيقة|يوم)', t):      # "منذ N ساعات" feed tail
        return True
    if re.search(r'[A-Za-z]{6,}', t):                              # long Latin run
        return True
    if re.search(r'الساعة\s+\d{1,2}:\d{2}\s*\(', t):             # timestamp
        return True
    if re.search(r'نشر (?:في|بتاريخ)\s*:', t):
        return True
    return False

h['text_clean'] = h['text'].apply(light_clean)
h['n_words_clean'] = h['text_clean'].apply(lambda t: len(t.split()))
h['has_artifact'] = h['text_clean'].apply(has_artifact)

before = h['source_domain'].value_counts()
h = h[~h['has_artifact']].reset_index(drop=True)     # REJECT artifact-bearing articles
after = h['source_domain'].value_counts()

print('rejected on artifact: kept %d of 15000 (%.0f%%)' % (len(h), 100*len(h)/15000))
print('\nsource distribution before -> after rejection:')
for src in before.index:
    print('  %-16s %5d -> %5d  (%.0f%% kept)' % (src, before[src], after.get(src, 0),
                                                 100*after.get(src, 0)/before[src]))

Q = '[\u0022\u00ab\u00bb\u201c\u201d]'
def frac(pat, col='text_clean'): return np.mean([bool(re.search(pat, t)) for t in h[col]])
print('\nbody punctuation preserved — digits %.0f%% | quotes %.0f%% | colon %.0f%%' %
      (100*frac(r'[0-9٠-٩]'), 100*frac(Q), 100*frac(':')))

rejected on artifact: kept 9148 of 15000 (61%)

source distribution before -> after rejection:
  aljazeera.net     7903 ->  4932  (62% kept)
  alaraby.co.uk     2731 ->  2317  (85% kept)
  arabi21.com       1259 ->   942  (75% kept)
  alarabiya.net     1117 ->   178  (16% kept)
  arabic.rt.com      670 ->    13  (2% kept)
  almayadeen.net     470 ->   343  (73% kept)
  maannews.net       341 ->    42  (12% kept)
  alquds.co.uk       217 ->   164  (76% kept)
  qudspress.com      182 ->   128  (70% kept)
  aa.com.tr           52 ->    39  (75% kept)
  wafa.ps             42 ->    39  (93% kept)
  aljazeera.com        8 ->     5  (62% kept)
  alaraby.tv           8 ->     6  (75% kept)

body punctuation preserved — digits 90% | quotes 92% | colon 61%


## Stage 2 — topic filtering (medium strictness)

Keep articles genuinely about politics / public affairs / political economy; drop culture, sports,
health, consumer-tech and pure opinion that a loose keyword match let through. Medium = require a
real density of political keywords AND that politics clearly outweighs off-topic signals, while
staying loose enough to land well above 3,500 (locally this leaves ~11,500).

In [3]:
POLITICS_CORE = [
    'الحكومة','الرئيس','الوزراء','البرلمان','الانتخابات','مجلس الأمن','الأمم المتحدة',
    'مفاوضات','اتفاق','قمة','عقوبات','الجيش','غارة','قصف','هدنة','وقف إطلاق النار',
    'المقاومة','الاحتلال','الاستيطان','مستوطن','حصار','عملية عسكرية','اشتباك','قتلى',
    'وزير','سفير','دبلوماسي','معاهدة','قرار','مبعوث','الخارجية','الدفاع','أمني',
    'انقلاب','ثورة','احتجاج','مظاهرة','اعتقال','أسرى','لاجئين','نازحين',
]
OFFTOPIC = [
    'الفيلم','السينما','المهرجان','الرواية','الشاعر','قصيدة','لوحة','معرض فني','الألبوم',
    'الدوري','المنتخب','مباراة','الهدف','اللاعب','بطولة','كأس',
    'وصفة','السعرات','التخسيس','البشرة','الوزن','الرجيم','أعراض','الحمية',
    'هاتف','تطبيق','الروبوت','يوتيوب','إنستغرام',
]

h['pol_score'] = h['text_clean'].apply(lambda t: sum(1 for k in POLITICS_CORE if k in t))
h['off_score'] = h['text_clean'].apply(lambda t: sum(1 for k in OFFTOPIC if k in t))

keep = (h['pol_score'] >= 3) & (h['pol_score'] > h['off_score'] * 2) & \
       (h['n_words_clean'] >= 400) & (h['n_words_clean'] <= 5000)
filt = h[keep].copy()
print('after topic filter:', len(filt), 'of', len(h))
print('pol_score mean %.1f | off_score mean %.1f' % (filt['pol_score'].mean(), filt['off_score'].mean()))

after topic filter: 7217 of 9148
pol_score mean 6.9 | off_score mean 0.4


## Stage 3 — dedup and select 3,500 (natural source distribution)

Content-level dedup on a normalized opening, then take 3,500 sampled within length strata so I
don't skew short or long. Source mix is left natural (no cap), per the decision.

In [4]:
def sig(t):
    s = re.sub(r'[^\u0600-\u06FF ]', '', t)      # arabic letters + space, for the dedup key
    return re.sub(r'\s+', ' ', s).strip()[:120]

filt['sig'] = filt['text_clean'].apply(sig)
filt = filt.drop_duplicates('sig').drop_duplicates('url').reset_index(drop=True)
print('after dedup:', len(filt))

TARGET = 3500
if len(filt) > TARGET:
    filt['len_bin'] = pd.qcut(filt['n_words_clean'], q=5, labels=False, duplicates='drop')
    final = (filt.groupby('len_bin', group_keys=False)
                  .apply(lambda g: g.sample(n=int(round(TARGET * len(g) / len(filt))),
                                            random_state=42))
                  .reset_index(drop=True))
    if len(final) > TARGET:
        final = final.sample(n=TARGET, random_state=42).reset_index(drop=True)
    elif len(final) < TARGET:
        extra = filt[~filt['url'].isin(final['url'])].sample(n=TARGET - len(final), random_state=42)
        final = pd.concat([final, extra]).reset_index(drop=True)
else:
    final = filt.copy()
    print('WARNING: fewer than', TARGET, 'available — relax the filter')

print('final:', len(final))

after dedup: 7212
final: 3500


/tmp/ipykernel_16/2945500508.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=int(round(TARGET * len(g) / len(filt))),


In [5]:
out = pd.DataFrame({
    'id': [f'HA_{i:05d}' for i in range(len(final))],
    'text': final['text_clean'].values,
    'length_words': final['n_words_clean'].values,
    'source_domain': final['source_domain'].values,
    'url': final['url'].values,
})
out.to_parquet('/kaggle/working/ha_corpus.parquet', index=False)
out.to_csv('/kaggle/working/ha_corpus.csv', index=False)
print('saved ha_corpus:', out.shape)

saved ha_corpus: (3500, 5)


In [6]:
print('=== final human corpus ===')
print('articles :', len(out))
print('length: mean %.0f | median %.0f | min %d | max %d' %
      (out['length_words'].mean(), out['length_words'].median(),
       out['length_words'].min(), out['length_words'].max()))
print('\nsource distribution (natural):')
print(out['source_domain'].value_counts())

Q = '[\u0022\u00ab\u00bb\u201c\u201d]'
def fr(pat): return np.mean([bool(re.search(pat, t)) for t in out['text']])
print('\n--- RAW confirmation (must stay high) ---')
print('has digits : %.0f%%' % (100*fr(r'[0-9٠-٩]')))
print('has quotes : %.0f%%' % (100*fr(Q)))
print('has colon  : %.0f%%' % (100*fr(':')))
print('mean digits/article: %.1f' % np.mean([len(re.findall(r'[0-9٠-٩]', t)) for t in out['text']]))

print('\n--- artifact confirmation (must all be 0) ---')
print('pipe (non-month):', sum(1 for t in out['text']
      if re.search(rf'\|(?!{MONTHS})', t) and not re.search(rf'{MONTHS}\|{MONTHS}', t)))
for kw in ('آخر تحديث', 'غرينتش', 'GMT', 'بتوقيت'):
    print('%s:' % kw, sum(1 for t in out['text'] if kw in t))
print('long latin:', sum(1 for t in out['text'] if re.search(r'[A-Za-z]{6,}', t)))

print('\n--- sample cleaned article ---')
print(out.iloc[0]['text'][:600])

=== final human corpus ===
articles : 3500
length: mean 751 | median 592 | min 400 | max 4990

source distribution (natural):
source_domain
aljazeera.net     1918
alaraby.co.uk      861
arabi21.com        365
almayadeen.net     140
qudspress.com       58
alarabiya.net       57
alquds.co.uk        47
aa.com.tr           18
maannews.net        16
wafa.ps             12
arabic.rt.com        4
aljazeera.com        2
alaraby.tv           2
Name: count, dtype: int64

--- RAW confirmation (must stay high) ---
has digits : 90%
has quotes : 92%
has colon  : 60%
mean digits/article: 22.3

--- artifact confirmation (must all be 0) ---
pipe (non-month): 0
آخر تحديث: 0
غرينتش: 0
GMT: 0
بتوقيت: 0
long latin: 0

--- sample cleaned article ---
بارنياع قال إنه لا حاجة لأحد في اليورانيوم المخصب لدرجة 60 في المئة- جيتي
وقال رئيس" الموساد" دافيد بارنياع: "إيران لن تملك سلاحا نوويا؛ لا في القريب العاجل ولا على المدى البعيد"، مضيفا: "هذا تعهد الموساد الذي سيبذل مع أجهزة الأمن الأخرى كل جهد مستطاع لإحباط هذا

## Notes

- **No body-level cleaning.** Stage 1 removes only scraping wrappers; the RAW confirmation proves
  digits/quotes/colons survive at high rates. This is the fix for the sanitized-corpus problem
  that would otherwise let a classifier cheat on punctuation.
- Natural source distribution kept on purpose (AlJazeera dominates because it is the largest slice
  of trusted long-form political coverage). I note this as a corpus characteristic in the thesis.
- Output schema (`id`, `text`, `length_words`, `source_domain`, `url`) matches what NB2b/NB2c and
  the pairing logic expect, so the rest of the pipeline runs unchanged on this new corpus.
- Upload as `aigt-ha-corpus` for the downstream notebooks.